In [10]:
# fit linear regression model and plot results
from statsmodels.formula.api import ols
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [2]:
csv_filename = "../CleanData/train_data_with_pca_05_09.csv"
df_train = pd.read_csv(csv_filename)
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72967 entries, 0 to 72966
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   PCA_0       72967 non-null  float64
 1   PCA_1       72967 non-null  float64
 2   PCA_2       72967 non-null  float64
 3   PCA_3       72967 non-null  float64
 4   PCA_4       72967 non-null  float64
 5   yearID      72967 non-null  int64  
 6   score_diff  72967 non-null  int64  
dtypes: float64(5), int64(2)
memory usage: 3.9 MB


In [4]:
csv_filename_test = "../CleanData/test_data_with_pca_05_09.csv"
df_test = pd.read_csv(csv_filename_test)
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18242 entries, 0 to 18241
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   PCA_0       18242 non-null  float64
 1   PCA_1       18242 non-null  float64
 2   PCA_2       18242 non-null  float64
 3   PCA_3       18242 non-null  float64
 4   PCA_4       18242 non-null  float64
 5   yearID      18242 non-null  int64  
 6   score_diff  18242 non-null  int64  
dtypes: float64(5), int64(2)
memory usage: 997.7 KB


In [8]:
cols = [col for col in df_train.columns if col != "score_diff"]

formula = f"score_diff ~ {' + '.join(cols)} + 0"
print(formula)
lr_model = ols(formula, data=df_train).fit()
# print(lr_model.params)
print(lr_model.summary())

score_diff ~ PCA_0 + PCA_1 + PCA_2 + PCA_3 + PCA_4 + yearID + 0
                                 OLS Regression Results                                
Dep. Variable:             score_diff   R-squared (uncentered):                   0.003
Model:                            OLS   Adj. R-squared (uncentered):              0.003
Method:                 Least Squares   F-statistic:                              36.36
Date:                Fri, 09 May 2025   Prob (F-statistic):                    3.03e-44
Time:                        13:04:20   Log-Likelihood:                     -2.1071e+05
No. Observations:               72967   AIC:                                  4.214e+05
Df Residuals:                   72961   BIC:                                  4.215e+05
Df Model:                           6                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|  

In [11]:
# Step 1: Calculate performance metrics
y_pred = lr_model.predict(df_test)
y_test = df_test['score_diff']
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

# Step 2: Print evaluation metrics
print(f"R² Score: {r2:.3f}")
print(f"Mean Squared Error (MSE): {mse:.3f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")
print(f"Mean Absolute Error (MAE): {mae:.3f}")

R² Score: 0.002
Mean Squared Error (MSE): 19.163
Root Mean Squared Error (RMSE): 4.378
Mean Absolute Error (MAE): 3.455


In [12]:
# try to predict a winner based on score_diff
df_test = df_test.assign(
    score_diff_pred = lr_model.predict(df_test)
)
correct_pred = 0
incorrect_pred = 0
for i, row in df_test.iterrows():
    if row['score_diff_pred'] > 0 and row['score_diff'] > 0:
        correct_pred += 1
    elif row['score_diff_pred'] < 0 and row['score_diff'] < 0:
        correct_pred += 1
    else:
        incorrect_pred += 1

print("Correct Predictions:", correct_pred, "Incorrect predictions:", incorrect_pred)
print("Percentage of correct pred (accuracy):", correct_pred / (correct_pred + incorrect_pred))


Correct Predictions: 9769 Incorrect predictions: 8473
Percentage of correct pred (accuracy): 0.5355224207871944
